In [1]:
import pandas as pd
from run_all_task import get_mixed_dataset
from classic_ml_func import model_gridsearch
from prepare_data import extract_mel_spectrogram, extract_mfcc, extract_lfcc, extract_cqcc, extract_gtcc
from ASV_dl_func import *

In [2]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

In [ ]:
from omegaconf import OmegaConf

config = OmegaConf.load("config.yaml")

la_metadata = config.datasets.LA.metadata
la_flac_folders = config.datasets.LA.flac
in_the_wild_dir = config.datasets.paths.in_the_wild_dir

In [ ]:
train_df, val_df, test_df = get_mixed_dataset(in_the_wild_dir, la_metadata, la_flac_folders)

In [22]:
train_aug = add_dataAugmentation(train_df)

In [ ]:
feature_extractors_map = {
    'cqcc': extract_cqcc,
    'gtcc': extract_gtcc,
    'mel-spect': extract_mel_spectrogram,
    'mfcc': extract_mfcc,
    'lfcc': extract_lfcc
}
col_name = feature_extractors_map.keys

In [8]:
train_df_prepared = extract_features(train_aug, feature_extractors_map)

   - Ekstrahuję: mel-spect


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:   15.8s
[Parallel(n_jobs=-1)]: Done 227 tasks      | elapsed:   16.7s
[Parallel(n_jobs=-1)]: Done 3680 tasks      | elapsed:   21.6s
[Parallel(n_jobs=-1)]: Done 9280 tasks      | elapsed:   27.3s
[Parallel(n_jobs=-1)]: Done 16480 tasks      | elapsed:   32.0s
[Parallel(n_jobs=-1)]: Done 25280 tasks      | elapsed:   37.4s
[Parallel(n_jobs=-1)]: Done 28600 out of 28631 | elapsed:   39.2s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done 28631 out of 28631 | elapsed:   39.2s finished


In [9]:
train_df_prepared = train_df_prepared.dropna(subset=col_name)

In [11]:

val_df = extract_features(val_df, feature_extractors_map)
val_df_prepared = val_df.dropna()

test_df = extract_features(test_df, feature_extractors_map)
test_df_prepared = test_df.dropna()

   - Ekstrahuję: mel-spect


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 1120 tasks      | elapsed:    1.1s
[Parallel(n_jobs=-1)]: Done 4535 tasks      | elapsed:    3.8s
[Parallel(n_jobs=-1)]: Done 4800 out of 4800 | elapsed:    4.0s finished


# Sprawdzenie zbalansowania

In [12]:
train_df_noscale = balance_func(train_df_prepared, col_name='label')

Zbilansowane dane: true=8279, false=8279


In [ ]:
val_df_noscale = balance_func(val_df_prepared, col_name='label')
test_df_noscale = balance_func(val_df_prepared, col_name='label')

In [ ]:
model_gridsearch(train_df_noscale, test_df_noscale, col_name)


=== FEATURE: cqcc ===
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Saved SVM & XGBoost results for feature: cqcc

=== FEATURE: gtcc ===
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Saved SVM & XGBoost results for feature: gtcc

=== FEATURE: mfcc ===
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Saved SVM & XGBoost results for feature: mfcc

=== FEATURE: lfcc ===
Fitting 5 folds for each of 9 candidates, totalling 45 fits
Fitting 5 folds for each of 12 candidates, totalling 60 fits
Saved SVM & XGBoost results for feature: lfcc
